# Pass 1 v4 — Reaction-Aware Chemical Name Resolver

**Full corrections from 120-compound stress test (4×30 chunks):**

1. Full archaic dictionary (56 acid→ester + 40+ archaic terms) from Pass 2
2. Greek prefix normalization (α→2, β→3, γ→4)
3. Typo/OCR correction dictionary
4. Smart comma parser — preserves position descriptors (2,4- / x,x- / 1,2,3,6-)
5. Sequential variant chaining (prefix → alias → -yl→-ol → resolve)
6. Sulfonamide guard (S(=O)(=O)N distinct from C(=O)N)
7. Leading parenthetical handling
8. Mercaptal di-X parser + oxime ester template + anilide handler
9. Embedded entity scanner
10. Mixture auto-retry via REACTION

## 1. Install & Upload

In [18]:
# RDKit install
import subprocess, sys
try:
    from rdkit import Chem
    print('RDKit already installed.')
except ImportError:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'rdkit-pypi', '-q'])
    except subprocess.CalledProcessError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'rdkit', '-q'])


import os, re, time, warnings, json
import pandas as pd
import numpy as np
from collections import Counter
warnings.filterwarnings("ignore")

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
import pubchempy as pcp

try:
    import cirpy
    HAS_CIRPY = True
except ImportError:
    HAS_CIRPY = False

print("Upload your dataset CSV:")
INPUT_FILE = "1947-King-USDA_Dataset.csv"
print(f"\nUploaded: {INPUT_FILE}")
print(f"CIRpy: {'YES' if HAS_CIRPY else 'NO'}")
print(f"RDKit: {Chem.rdBase.rdkitVersion}")

RDKit already installed.
Upload your dataset CSV:

Uploaded: 1947-King-USDA_Dataset.csv
CIRpy: YES
RDKit: 2025.09.2


## 2. Load & Configure

In [19]:
NAME_COL = "Chemical"  #@param {type:"string"}

df = pd.read_csv(INPUT_FILE, sep=None, engine='python')
print(f"Loaded: {len(df)} rows")

if NAME_COL not in df.columns:
    candidates = [c for c in df.columns if 'chem' in c.lower() or 'name' in c.lower() or 'material' in c.lower()]
    if candidates:
        NAME_COL = candidates[0]
        print(f"Auto-detected: '{NAME_COL}'")
    else:
        print(f"ERROR: Set NAME_COL. Columns: {list(df.columns)}")

df["_name"] = df[NAME_COL].astype(str).str.strip()
print(f"Name column: '{NAME_COL}'")

Loaded: 7089 rows
Name column: 'Chemical'


## 3. Dictionaries (Archaic, Typo, Alias, Prefix)

In [20]:
# ================================================================
# FIX #3: TYPO / OCR CORRECTION DICTIONARY
# Applied to raw names before any parsing
# ================================================================
TYPO_CORRECTIONS = {
    "benzoid": "benzoic",
    "anyl": "amyl",
    "pentacrythritol": "pentaerythritol",
    "pentacrythrityl": "pentaerythrityl",
    "endonethanol": "endomethanol",
    "l-oic": "1-oic",
    "tetra-hydro": "tetrahydro",
    "di-n-": "di-n",
    "iso-bornyl": "isobornyl",
    "iso-amyl": "isoamyl",
    "iso-butyl": "isobutyl",
    "iso-propyl": "isopropyl",
}

def fix_typos(name):
    """Apply known typo corrections to a name."""
    fixed = name
    for typo, correction in TYPO_CORRECTIONS.items():
        fixed = re.sub(re.escape(typo), correction, fixed, flags=re.IGNORECASE)
    return fixed


# ================================================================
# FIX #2: GREEK PREFIX NORMALIZATION
# ================================================================
GREEK_PREFIXES = {
    'alpha-': '2-', 'α-': '2-',
    'beta-': '3-', 'β-': '3-',
    'gamma-': '4-', 'γ-': '4-',
    'delta-': '5-', 'δ-': '5-',
    'omega-': 'ω-',  # keep as-is, context-dependent
}

# ================================================================
# FIX #1: FULL ARCHAIC DICTIONARY (from Pass 2)
# ================================================================
ARCHAIC_DICT = {
    "mercaptal": "dithioacetal", "mercaptol": "dithioketal",
    "mercaptan": "thiol", "carbinol": "methanol",
    "pinacol": "2,3-dimethyl-2,3-butanediol",
    "furfurol": "furfural", "formol": "formaldehyde",
    "chloral": "trichloroacetaldehyde",
    "paraldehyde": "2,4,6-trimethyl-1,3,5-trioxane",
    "cellosolve": "2-ethoxyethanol",
    "carbitol": "2-(2-ethoxyethoxy)ethanol",
    "dowanol": "propylene glycol methyl ether",
    "dioxane": "1,4-dioxane", "dioxan": "1,4-dioxane",
    "trioxane": "1,3,5-trioxane",
    "picoline": "methylpyridine",
    "collidine": "2,4,6-trimethylpyridine",
    "lutidine": "dimethylpyridine",
    "xylidine": "dimethylaniline",
    "toluidine": "methylaniline",
    "anisole": "methoxybenzene", "phenetole": "ethoxybenzene",
    "cresol": "methylphenol", "xylenol": "dimethylphenol",
    "thymol": "2-isopropyl-5-methylphenol",
    "guaiacol": "2-methoxyphenol",
    "catechol": "1,2-benzenediol",
    "resorcinol": "1,3-benzenediol",
    "hydroquinone": "1,4-benzenediol",
    "pyrogallol": "1,2,3-benzenetriol",
    "orcinol": "3,5-dihydroxytoluene",
    "vanillin": "4-hydroxy-3-methoxybenzaldehyde",
    "coumarin": "2H-chromen-2-one",
    "acrolein": "2-propenal",
    "crotonaldehyde": "2-butenal",
    "mesityl oxide": "4-methylpent-3-en-2-one",
    "diacetone alcohol": "4-hydroxy-4-methylpentan-2-one",
    "pentaerythritol": "2,2-bis(hydroxymethyl)-1,3-propanediol",
    # Archaic acid names
    "undecylic acid": "undecanoic acid",
    "undecylenic acid": "undecylenic acid",
    "pelargonic acid": "nonanoic acid",
    "enanthic acid": "heptanoic acid",
    "margaric acid": "heptadecanoic acid",
    "acetolactic acid": "2-acetyllactic acid",
    "xanthoacetic acid": "(ethylthio)thiocarbonylacetic acid",
    "ethylxanthoacetic acid": "ethyl xanthoacetate",
}

# ================================================================
# ACID → ESTER SUFFIX (from Pass 2)
# ================================================================
ACID_TO_ESTER = {
    "acetic": "acetate", "propionic": "propionate",
    "butyric": "butyrate", "valeric": "valerate",
    "caproic": "caproate", "caprylic": "caprylate",
    "capric": "caprate", "lauric": "laurate",
    "myristic": "myristate", "palmitic": "palmitate",
    "stearic": "stearate", "oleic": "oleate",
    "linoleic": "linoleate", "benzoic": "benzoate",
    "salicylic": "salicylate", "phthalic": "phthalate",
    "succinic": "succinate", "citric": "citrate",
    "tartaric": "tartrate", "oxalic": "oxalate",
    "malonic": "malonate", "maleic": "maleate",
    "fumaric": "fumarate", "glutaric": "glutarate",
    "adipic": "adipate", "sebacic": "sebacate",
    "azelaic": "azelate", "cinnamic": "cinnamate",
    "formic": "formate", "carbonic": "carbonate",
    "lactic": "lactate", "glycolic": "glycolate",
    "mandelic": "mandelate", "crotonic": "crotonate",
    "undecylenic": "undecylenate", "abietic": "abietate",
    "levulinic": "levulinate", "pyruvic": "pyruvate",
    "phosphoric": "phosphate", "sulfuric": "sulfate",
    "nitric": "nitrate", "boric": "borate",
    "thioglycolic": "thioglycolate",
    "phenylacetic": "phenylacetate",
    "toluic": "toluate", "nicotinic": "nicotinate",
    "isobutyric": "isobutyrate", "isovaleric": "isovalerate",
    "pelargonic": "pelargonate", "enanthic": "enanthate",
    "heptanoic": "heptanoate", "chloroacetic": "chloroacetate",
    "dichloroacetic": "dichloroacetate",
    "trichloroacetic": "trichloroacetate",
    "cyanoacetic": "cyanoacetate",
    "acetoacetic": "acetoacetate",
    "chlorobenzoic": "chlorobenzoate",
    "aminobenzoic": "aminobenzoate",
    "nitrobenzoic": "nitrobenzoate",
}

# ================================================================
# COMPONENT ALIASES (FIX #5: expanded -yl → alcohol)
# ================================================================
COMPONENT_ALIASES = {
    # -yl → alcohol / parent compound
    'methyl': 'methanol', 'ethyl': 'ethanol',
    'propyl': 'propanol', 'butyl': 'butanol',
    'amyl': 'pentanol', 'hexyl': 'hexanol',
    'heptyl': 'heptanol', 'octyl': 'octanol',
    'nonyl': 'nonanol', 'decyl': 'decanol',
    'undecyl': 'undecanol', 'dodecyl': 'dodecanol',
    'lauryl': 'dodecanol', 'cetyl': 'hexadecanol',
    'stearyl': 'octadecanol', 'oleyl': 'oleyl alcohol',
    'benzyl': 'benzyl alcohol', 'phenyl': 'phenol',
    'cyclohexyl': 'cyclohexanol',
    'bornyl': 'borneol', 'isobornyl': 'isoborneol',
    'fenchyl': 'fenchol', 'menthyl': 'menthol',
    'geranyl': 'geraniol', 'linalyl': 'linalool',
    'citronellyl': 'citronellol', 'nerol': 'nerol',
    'cinnamyl': 'cinnamyl alcohol',
    'furfuryl': 'furfuryl alcohol',
    'tetrahydrofurfuryl': 'tetrahydrofurfuryl alcohol',
    'glyceryl': 'glycerol', 'glycol': 'ethylene glycol',
    'cresyl': 'cresol', 'tolyl': 'cresol',
    'naphthyl': 'naphthol', 'allyl': 'allyl alcohol',
    'phenethyl': '2-phenylethanol',
    'anilino': 'aniline', 'anilide': 'aniline',
    # Substituted phenyl → phenol
    'allyphenyl': 'allylphenol', 'allylphenyl': 'allylphenol',
    'amylphenyl': 'amylphenol',
    'benzhydryl': 'benzhydrol',
    # Multi-carbon named groups
    'isoamyl': 'isoamyl alcohol',
    'isobutyl': 'isobutyl alcohol',
    'isopropyl': 'isopropanol',
    'tert-butyl': 'tert-butanol',
    'sec-butyl': 'sec-butanol',
    'n-amyl': 'pentanol', 'n-butyl': 'butanol',
    'n-propyl': 'propanol', 'n-hexyl': 'hexanol',
    'n-octyl': 'octanol', 'n-decyl': 'decanol',
    # Thiols for mercaptal
    'dioctyl': 'octanethiol',
    'dibutyl': 'butanethiol',
    'diethyl': 'ethanethiol',
}

print(f"Dictionaries loaded:")
print(f"  Typo corrections: {len(TYPO_CORRECTIONS)}")
print(f"  Greek prefixes: {len(GREEK_PREFIXES)}")
print(f"  Archaic terms: {len(ARCHAIC_DICT)}")
print(f"  Acid→ester: {len(ACID_TO_ESTER)}")
print(f"  Component aliases: {len(COMPONENT_ALIASES)}")

Dictionaries loaded:
  Typo corrections: 12
  Greek prefixes: 9
  Archaic terms: 46
  Acid→ester: 59
  Component aliases: 57


## 4. Route Classifier + Entity Scanner

In [21]:
KNOWN_ENTITIES = {
    "acetaldehyde", "benzaldehyde", "formaldehyde", "butyraldehyde",
    "propionaldehyde", "valeraldehyde", "crotonaldehyde", "furfural",
    "acrolein", "citronellal", "cinnamaldehyde",
    "acetone", "cyclohexanone", "acetophenone", "camphor",
    "benzophenone",
    "methanol", "ethanol", "propanol", "butanol", "pentanol",
    "hexanol", "octanol", "decanol", "dodecanol",
    "glycerol", "glycol", "ethylene glycol", "diethylene glycol",
    "propylene glycol", "sorbitol", "mannitol", "xylitol",
    "geraniol", "linalool", "menthol", "borneol", "fenchol",
    "phenol", "cresol", "catechol", "resorcinol", "hydroquinone",
    "naphthol", "thymol", "guaiacol", "eugenol",
    "acetic acid", "propionic acid", "butyric acid", "valeric acid",
    "caprylic acid", "lauric acid", "palmitic acid", "stearic acid",
    "oleic acid", "benzoic acid", "salicylic acid", "phthalic acid",
    "succinic acid", "adipic acid", "citric acid", "lactic acid",
    "oxalic acid", "abietic acid", "maleic acid", "cinnamic acid",
    "carbonic acid", "cyanoacetic acid", "mandelic acid",
    "aniline", "morpholine", "piperidine", "pyrrolidine",
}

DERIVATIVE_SUFFIXES = [
    "oate", "anoate", "ate",
    "amide", "anilide", "ide",
    "acetal", "mercaptal",
    "yl", "oyl", "oxy", "ido", "ato",
]

PRODUCT_TYPE_PATTERNS = [
    (r'\b(tri|di|mono)?ester\b', 'ester', True),
    (r'\bamide\b|\banilide\b|\bformamide\b|\bacetamide\b|\bsulfonamide\b', 'amide', True),
    (r'\banhydride\b', 'anhydride', True),
    (r'\bsalt\b', 'salt', True),
    (r'\bacetal\b', 'acetal', False),
    (r'\bmercaptal\b|\bdithioacetal\b', 'mercaptal', False),
    (r'\bether\b', 'ether', False),
    (r'\bimide\b|\bimido\b', 'imide', True),
    (r'\boxime\b', 'oxime_standalone', False),
    (r'\bhydrazone\b', 'hydrazone', False),
    (r'\bsemicarbazone\b', 'semicarbazone', False),
    (r'\bthioester\b', 'thioester', True),
]


def scan_embedded_entities(name):
    low = name.lower().strip()
    found = []
    for entity in sorted(KNOWN_ENTITIES, key=len, reverse=True):
        if entity in low:
            pattern = r'\b' + re.escape(entity) + r'\b'
            if re.search(pattern, low):
                found.append(entity)
    suffix_hits = []
    words = re.findall(r'[a-z]+', low)
    for word in words:
        for suffix in DERIVATIVE_SUFFIXES:
            if word.endswith(suffix) and len(word) > len(suffix) + 2:
                suffix_hits.append(f"{word} (-{suffix})")
                break
    return found, suffix_hits


def is_irregular(name):
    low = name.lower()
    # FIX #8: Tighter irregular checks
    if re.search(r'\bmixed\b|\bmixture\b|\bblend\b', low):
        return True, "MIXTURE"
    if re.search(r'\bnaphtha\b|\bpetroleum\b|\bfraction\b|\bkerosene\b', low):
        # Don't flag if it also has chemical structure keywords
        if not re.search(r'acid|ester|amine|ether|alcohol', low):
            return True, "PETROLEUM"
    if re.search(r'\bmineral\b', low) and not re.search(r'acid|salt', low):
        return True, "MINERAL"
    if re.search(r'polymer|polyethylene|\bresin\b|\bwax\b|\bcellulose\b|\bstarch\b', low):
        # Allow polyester as it's a reaction product type
        if not re.search(r'polyester', low):
            return True, "POLYMER"
    if re.search(r'\badduct\b|\bcondensate?\b|\bcooxidation\b|\bcrude product\b', low):
        return True, "REACTION_PRODUCT"
    if re.search(r'\boil\b', low) and not re.search(r'from\s+.*oil|acid|ester|alcohol', low):
        return True, "OIL"
    if len(name) <= 3:
        return True, "ABBREVIATION"
    if not name or name.lower() in ['nan', 'n/a', '', 'none', 'unknown']:
        return True, "EMPTY"
    return False, None


def classify_route(name):
    raw = str(name).strip()
    # Apply typo fix before classification
    raw_fixed = fix_typos(raw)
    low = raw_fixed.lower()

    irreg, reason = is_irregular(raw_fixed)
    if irreg:
        return 'SKIP', {'reason': reason}

    for pattern, ptype, needs_acid in PRODUCT_TYPE_PATTERNS:
        if re.search(pattern, low):
            return 'REACTION', {'product_type': ptype, 'needs_acid': needs_acid}

    if ',' in raw_fixed and 'acid' in low:
        return 'REACTION', {'product_type': 'unknown_comma_acid', 'needs_acid': True}

    entities, suffixes = scan_embedded_entities(raw_fixed)
    if len(entities) >= 2:
        return 'REACTION', {'product_type': 'entity_scan_detected', 'entities_found': entities}
    if len(entities) >= 1 and len(suffixes) >= 1:
        return 'REACTION', {'product_type': 'suffix_scan_detected', 'entities_found': entities, 'suffixes_found': suffixes}

    return 'SIMPLE', {'entities': entities, 'suffixes': suffixes}


# Classify
routes = df["_name"].apply(classify_route)
df["_route"] = routes.apply(lambda x: x[0])
df["_route_details"] = routes.apply(lambda x: x[1])

print("ROUTE ASSIGNMENT:")
for route, count in df["_route"].value_counts().items():
    print(f"  {route:15s} {count:5d}  ({count/len(df)*100:.1f}%)")

reaction_rows = df[df["_route"] == "REACTION"]
ptypes = reaction_rows["_route_details"].apply(lambda d: d.get('product_type', 'unknown'))
print(f"\nREACTION product types:")
for pt, count in ptypes.value_counts().items():
    print(f"    {pt:35s} {count:5d}")

ROUTE ASSIGNMENT:
  REACTION         3617  (51.0%)
  SIMPLE           3261  (46.0%)
  SKIP              211  (3.0%)

REACTION product types:
    ester                                2686
    ether                                 329
    suffix_scan_detected                  246
    unknown_comma_acid                     74
    acetal                                 63
    salt                                   55
    entity_scan_detected                   47
    amide                                  45
    oxime_standalone                       27
    semicarbazone                          15
    anhydride                              15
    imide                                  12
    mercaptal                               3


## 5. Smart Comma Parser (Fixed)

In [22]:
SUB_DERIVATIZATIONS = {
    "oxime": {"base_pattern": r'^(.+?)\s+oxime$', "template": "[CX3:1]=[O:2]>>[CX3:1]=[N][OH]"},
    "hydrazone": {"base_pattern": r'^(.+?)\s+hydrazone$', "template": "[CX3:1]=[O:2]>>[CX3:1]=[N][NH2]"},
    "semicarbazone": {"base_pattern": r'^(.+?)\s+semicarbazone$', "template": "[CX3:1]=[O:2]>>[CX3:1]=[N]NC(=O)N"},
    "cyanohydrin": {"base_pattern": r'^(.+?)\s+cyanohydrin$', "template": "[CH1:1]=[O:2]>>[C:1]([OH])C#N"},
}

MULTIPLICITY = {'mono': 1, 'di': 2, 'tri': 3, 'tetra': 4, 'penta': 5, 'hexa': 6, 'poly': -1}


def find_structural_comma(name):
    """
    FIX #4: Find the 'real' comma that separates A from X Y.
    Skip commas inside position descriptors like 2,4- or 1,2,3,6- or x,x-.
    """
    # Remove parenthetical content temporarily for comma finding
    clean = re.sub(r'\([^)]*\)', lambda m: '~' * len(m.group()), name)

    for i, ch in enumerate(clean):
        if ch == ',':
            # Check if this comma is inside a position descriptor
            # Look at surrounding context
            before = clean[:i]
            after = clean[i+1:]

            # Pattern: digit,digit or letter,letter (position descriptor)
            # e.g., "2,4-di" or "x,x-di" or "1,2,3,6-tetra"
            if (re.search(r'[\dx]$', before.rstrip()) and
                re.search(r'^\s*[\dx]', after)):
                continue  # Skip — this is a position descriptor comma

            return i  # This is the structural comma

    return -1  # No structural comma found


def parse_AXY(name):
    clean = name.strip()

    result = {
        'parent_A': None,
        'modifier_X': None,
        'product_type_Y': None,
        'parenthetical': None,
        'x_sub_derivatization': None,
        'x_base_compound': None,
        'multiplicity': 1,
        'parse_confidence': None,
        'status': None
    }

    # === 1. MERCAPTAL SPECIAL CASE ===
    mercaptal_match = re.match(
        r'^(.+?)\s+di(\w+)\s+mercaptal\s*$', clean, re.IGNORECASE
    )
    if mercaptal_match:
        result['parent_A'] = mercaptal_match.group(1).strip()
        result['modifier_X'] = mercaptal_match.group(2).strip()
        result['product_type_Y'] = 'mercaptal'
        result['multiplicity'] = 2
        result['parse_confidence'] = 'HIGH'
        result['status'] = 'OK'
        return result

    # === 2. COMMA FORMAT ===
    comma_pos = find_structural_comma(clean)
    if comma_pos >= 0:
        a_part = clean[:comma_pos].strip()
        xy_part = clean[comma_pos+1:].strip()
        result['parent_A'] = a_part

        for pattern, ptype, _ in PRODUCT_TYPE_PATTERNS:
            multi_match = re.search(
                r'(mono|di|tri|tetra|penta|hexa|poly)?' + pattern,
                xy_part, re.IGNORECASE
            )
            if multi_match:
                result['product_type_Y'] = ptype
                mp = multi_match.group(1)
                if mp:
                    result['multiplicity'] = MULTIPLICITY.get(mp.lower(), 1)
                x_part = xy_part[:multi_match.start()].strip().rstrip(',')
                if x_part:
                    result['modifier_X'] = x_part
                result['parse_confidence'] = 'HIGH'
                result['status'] = 'OK'
                return result

        result['modifier_X'] = xy_part
        result['product_type_Y'] = 'unknown'
        result['parse_confidence'] = 'LOW'
        result['status'] = 'OK'
        return result

    # === 3. NO-COMMA FORMAT ===
    for pattern, ptype, _ in PRODUCT_TYPE_PATTERNS:
        full_match = re.match(
            r'^(.+?)\s+(.+?)\s+' + pattern + r'\s*$',
            clean,
            re.IGNORECASE
        )
        if full_match:
            result['parent_A'] = full_match.group(1).strip()
            result['modifier_X'] = full_match.group(2).strip()
            result['product_type_Y'] = ptype
            result['parse_confidence'] = 'MEDIUM'
            result['status'] = 'OK'
            break

    # === 4. END-MATCH FORMAT ===
    if result['parent_A'] is None:
        for pattern, ptype, _ in PRODUCT_TYPE_PATTERNS:
            end_match = re.search(pattern + r'\s*$', clean, re.IGNORECASE)
            if end_match:
                before = clean[:end_match.start()].strip()
                words = before.rsplit(None, 1)
                if len(words) == 2:
                    result['parent_A'] = words[0].strip()
                    result['modifier_X'] = words[1].strip()
                elif len(words) == 1:
                    result['parent_A'] = words[0].strip()
                result['product_type_Y'] = ptype
                result['parse_confidence'] = 'LOW'
                result['status'] = 'OK'
                break

    # === 5. SUB-DERIVATIZATION ===
    if result['modifier_X']:
        for sub_name, sub_info in SUB_DERIVATIZATIONS.items():
            sub_match = re.match(sub_info['base_pattern'], result['modifier_X'], re.IGNORECASE)
            if sub_match:
                result['x_sub_derivatization'] = sub_name
                result['x_base_compound'] = sub_match.group(1).strip()
                break

    # === 6. FINAL FALLBACK ===
    if result['parent_A'] is None and result['modifier_X'] is None:
        result['status'] = 'NO_MATCH'
    else:
        if result['status'] is None:
            result['status'] = 'OK'

    return result

# Test
test_parses = [
    "Acetic acid, butyl ester",
    "2,4-Dimethyl-1,2,3,6-tetrahydrobenzaldehyde ethanol acetal",
    "Benzyl x,x-diethylphenyl ether",
    "Butyric acid, anilide",
    "Acetaldehyde dioctyl mercaptal",
    "Benzoic acid, iso-butyraldehyde oxime ester",
    "(1-Methyl-1-butenyl)propyl-cyanoacetic acid,ethyl ester",
]

print("Parser test (v4):")
print("=" * 70)
for name in test_parses:
    p = parse_AXY(name)
    print(f"\n  {name}")
    print(f"    A={p['parent_A']}  X={p['modifier_X']}  Y={p['product_type_Y']}  mult={p['multiplicity']}  conf={p['parse_confidence']}")
    if p['x_sub_derivatization']:
        print(f"    sub-deriv: {p['x_sub_derivatization']} (base: {p['x_base_compound']})")

Parser test (v4):

  Acetic acid, butyl ester
    A=Acetic acid  X=butyl  Y=ester  mult=1  conf=HIGH

  2,4-Dimethyl-1,2,3,6-tetrahydrobenzaldehyde ethanol acetal
    A=2,4-Dimethyl-1,2,3,6-tetrahydrobenzaldehyde  X=ethanol  Y=acetal  mult=1  conf=MEDIUM

  Benzyl x,x-diethylphenyl ether
    A=Benzyl  X=x,x-diethylphenyl  Y=ether  mult=1  conf=MEDIUM

  Butyric acid, anilide
    A=Butyric acid  X=None  Y=amide  mult=1  conf=HIGH

  Acetaldehyde dioctyl mercaptal
    A=Acetaldehyde  X=octyl  Y=mercaptal  mult=2  conf=HIGH

  Benzoic acid, iso-butyraldehyde oxime ester
    A=Benzoic acid  X=iso-butyraldehyde oxime  Y=ester  mult=1  conf=HIGH

  (1-Methyl-1-butenyl)propyl-cyanoacetic acid,ethyl ester
    A=(1-Methyl-1-butenyl)propyl-cyanoacetic acid  X=ethyl  Y=ester  mult=1  conf=HIGH


In [17]:
import inspect
print(inspect.getsource(parse_AXY))


def parse_AXY(name):
    clean = name.strip()

    result = {
        'parent_A': None,
        'modifier_X': None,
        'product_type_Y': None,
        'parenthetical': None,
        'x_sub_derivatization': None,
        'x_base_compound': None,
        'multiplicity': 1,
        'parse_confidence': None,
        'status': None
    }

    # === 1. MERCAPTAL SPECIAL CASE ===
    mercaptal_match = re.match(
        r'^(.+?)\s+di(\w+)\s+mercaptal\s*$', clean, re.IGNORECASE
    )
    if mercaptal_match:
        result['parent_A'] = mercaptal_match.group(1).strip()
        result['modifier_X'] = mercaptal_match.group(2).strip()
        result['product_type_Y'] = 'mercaptal'
        result['multiplicity'] = 2
        result['parse_confidence'] = 'HIGH'
        result['status'] = 'OK'
        return result

    # === 2. COMMA FORMAT ===
    comma_pos = find_structural_comma(clean)
    if comma_pos >= 0:
        a_part = clean[:comma_pos].strip()
        xy_part = clean[comma_pos+1:].strip

## 10. Batch Processing

In [26]:
print(f"Processing {len(df)} compounds...")
print(f"  SIMPLE:   {(df['_route']=='SIMPLE').sum()}")
print(f"  REACTION: {(df['_route']=='REACTION').sum()}")
print(f"  SKIP:     {(df['_route']=='SKIP').sum()}")
print()

results = []
counters = Counter()
batch_start = time.time()

for idx, row in df.iterrows():
    name = row['_name']
    route = row['_route']
    details = row['_route_details']
    res = resolve_compound(name, route, details)
    results.append(res)
    counters[res['status']] += 1

    if (idx + 1) % 100 == 0 or idx == 0:
        elapsed = time.time() - batch_start
        rate = (idx + 1) / elapsed if elapsed > 0 else 0
        n_res = sum(1 for r in results if r['smiles'] is not None)
        remaining = (len(df) - idx - 1) / rate / 60 if rate > 0 else 0
        print(f"  [{idx+1:5d}/{len(df)}] Resolved: {n_res}  Rate: {rate:.1f}/sec  ETA: {remaining:.1f} min")

    if (idx + 1) % 200 == 0:
        save_cache(resolve_cache)
    if route != 'SKIP':
        time.sleep(0.15)

save_cache(resolve_cache)
elapsed_total = (time.time() - batch_start) / 60
print(f"\n{'='*60}")
print(f"  COMPLETE in {elapsed_total:.1f} minutes")
print(f"{'='*60}")
for status, count in counters.most_common():
    print(f"  {status:35s} {count:5d}  ({count/len(df)*100:.1f}%)")

Processing 7089 compounds...
  SIMPLE:   3261
  REACTION: 3617
  SKIP:     211

  [    1/7089] Resolved: 1  Rate: 198.4/sec  ETA: 0.6 min


AttributeError: 'NoneType' object has no attribute 'strip'

## 6. Component Resolver (Sequential Chaining)

In [12]:
CACHE_FILE = "resolver_cache_v4.json"

def load_cache():
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, 'r') as f:
            return json.load(f)
    return {}

def save_cache(cache):
    with open(CACHE_FILE, 'w') as f:
        json.dump(cache, f)

resolve_cache = load_cache()
print(f"Cache: {len(resolve_cache)} entries")


def generate_component_variants(name):
    """
    FIX #5: Sequential chaining — each transform feeds the next.
    Order: typo fix → Greek prefix → archaic sub → positional prefix → alias → -yl→-ol
    """
    raw = str(name).strip()
    low = raw.lower()
    variants = [(raw, 'original')]

    # Stage 1: Typo fix
    fixed = fix_typos(raw)
    if fixed != raw:
        variants.append((fixed, 'typo_fix'))

    # Stage 2: Greek prefix normalization
    current = fixed
    for greek, num in GREEK_PREFIXES.items():
        if current.lower().startswith(greek):
            current = num + current[len(greek):]
            variants.append((current, f'greek_{greek}'))
            break

    # Stage 3: Archaic term substitution
    for old_term, new_term in ARCHAIC_DICT.items():
        if old_term.lower() in current.lower():
            replaced = re.sub(re.escape(old_term), new_term, current, flags=re.IGNORECASE)
            variants.append((replaced, f'archaic_{old_term}'))

    # Stage 4: Positional prefix normalization (o-→2-, p-→4-, etc.)
    prefix_map = {'o-': '2-', 'm-': '3-', 'p-': '4-', 'n-': '', 'sec-': '', 'tert-': 'tert-', 'dl-': '', 'd-': '', 'l-': ''}
    for prefix, repl in prefix_map.items():
        # At start of string
        if current.lower().startswith(prefix):
            stripped = repl + current[len(prefix):]
            if stripped != current:
                variants.append((stripped.strip(), f'prefix_{prefix}'))
        # Within string (subword)
        patt = r'\b' + re.escape(prefix)
        if re.search(patt, current, re.IGNORECASE):
            subword = re.sub(patt, repl, current, flags=re.IGNORECASE).strip()
            if subword != current and subword not in [v for v, _ in variants]:
                variants.append((subword, f'subword_{prefix}'))

    # Stage 5: Alias lookup (SEQUENTIAL — applied to each variant so far)
    new_variants = []
    for v, strat in variants:
        v_low = v.lower().strip()
        if v_low in COMPONENT_ALIASES:
            aliased = COMPONENT_ALIASES[v_low]
            new_variants.append((aliased, f'{strat}+alias'))
        # Also try after stripping prefix
        for prefix in ['n-', 'o-', 'p-', 'm-', 'iso-', 'sec-', 'tert-']:
            if v_low.startswith(prefix):
                stripped_low = v_low[len(prefix):]
                if stripped_low in COMPONENT_ALIASES:
                    new_variants.append((COMPONENT_ALIASES[stripped_low], f'{strat}+strip_{prefix}+alias'))
    variants.extend(new_variants)

    # Stage 6: -yl → -ol conversion
    for v, strat in list(variants):
        if v.lower().endswith('yl') and v.lower() not in COMPONENT_ALIASES:
            ol_form = v[:-2] + 'ol'
            variants.append((ol_form, f'{strat}+yl_to_ol'))
            variants.append((v + ' alcohol', f'{strat}+add_alcohol'))
        # Handle substituted phenyl → phenol
        if 'phenyl' in v.lower():
            phenol = re.sub(r'phenyl', 'phenol', v, flags=re.IGNORECASE)
            variants.append((phenol, f'{strat}+phenyl_to_phenol'))

    # Strip parentheses
    stripped = re.sub(r'\s*\([^)]*\)', '', raw).strip()
    if stripped != raw:
        variants.append((stripped, 'strip_parens'))

    # Deduplicate
    seen = set()
    unique = []
    for v, s in variants:
        vc = v.strip()
        if vc and vc.lower() not in seen and len(vc) > 1:
            seen.add(vc.lower())
            unique.append((vc, s))

    return unique


def resolve_name_to_smiles(name, cache=None, max_retries=2):
    if cache is None:
        cache = resolve_cache

    key = name.lower().strip()
    if key in cache:
        entry = cache[key]
        return entry.get('smiles'), entry.get('source', 'cache')

    variants = generate_component_variants(name)

    for variant, strategy in variants:
        for attempt in range(max_retries):
            try:
                results = pcp.get_compounds(variant, 'name')
                if results and results[0].canonical_smiles:
                    smi = results[0].canonical_smiles
                    cache[key] = {'smiles': smi, 'source': f'pubchem_{strategy}'}
                    return smi, f'pubchem_{strategy}'
                break
            except pcp.PubChemHTTPError:
                time.sleep(1.5)
            except:
                break

        if HAS_CIRPY:
            try:
                smi = cirpy.resolve(variant, 'smiles')
                if smi:
                    cache[key] = {'smiles': smi, 'source': f'cirpy_{strategy}'}
                    return smi, f'cirpy_{strategy}'
            except:
                pass

    cache[key] = {'smiles': None, 'source': 'failed'}
    return None, 'failed'


# Test variant chaining
print("\nVariant test: 'o-allyphenyl'")
for v, s in generate_component_variants('o-allyphenyl'):
    print(f"  [{s}] {v}")

print("\nVariant test: 'alpha-Allylacetoacetic acid'")
for v, s in generate_component_variants('alpha-Allylacetoacetic acid'):
    print(f"  [{s}] {v}")

print("\nVariant test: '2-methyl-cyclohexyl'")
for v, s in generate_component_variants('2-methyl-cyclohexyl'):
    print(f"  [{s}] {v}")

Cache: 0 entries

Variant test: 'o-allyphenyl'
  [original] o-allyphenyl
  [prefix_o-] 2-allyphenyl
  [original+strip_o-+alias] allylphenol
  [original+yl_to_ol] o-allyphenol
  [original+add_alcohol] o-allyphenyl alcohol
  [prefix_o-+yl_to_ol] 2-allyphenol
  [prefix_o-+add_alcohol] 2-allyphenyl alcohol

Variant test: 'alpha-Allylacetoacetic acid'
  [original] alpha-Allylacetoacetic acid
  [greek_alpha-] 2-Allylacetoacetic acid

Variant test: '2-methyl-cyclohexyl'
  [original] 2-methyl-cyclohexyl
  [original+yl_to_ol] 2-methyl-cyclohexol
  [original+add_alcohol] 2-methyl-cyclohexyl alcohol


## 7. Reaction Templates

In [23]:
REACTION_TEMPLATES = {
    "ester": {
        "smarts": "[C:1](=[O:2])[OH:3].[OH:4][C:5]>>[C:1](=[O:2])[O:4][C:5]",
        "water_loss": 18.02, "product_check": "[CX3](=O)[OX2][C]",
        "a_requires": "[CX3](=O)[OH]", "x_requires": "[OH]",
    },
    "oxime_ester": {
        "smarts": "[C:1](=[O:2])[OH:3].[OH:4][N:5]=[C:6]>>[C:1](=[O:2])[O:4][N:5]=[C:6]",
        "water_loss": 18.02, "product_check": "[CX3](=O)[O][N]",
        "a_requires": "[CX3](=O)[OH]", "x_requires": "[OX2H1][NX2]",
    },
    "amide": {
        "smarts": "[C:1](=[O:2])[OH:3].[NH2:4][C:5]>>[C:1](=[O:2])[NH:4][C:5]",
        "water_loss": 18.02, "product_check": "[CX3](=O)[NX3]",
        "a_requires": "[CX3](=O)[OH]", "x_requires": "[NX3;H2,H1]",
    },
    "anhydride": {
        "smarts": "[C:1](=[O:2])[OH:3].[C:4](=[O:5])[OH:6]>>[C:1](=[O:2])[O][C:4](=[O:5])",
        "water_loss": 18.02, "product_check": "[CX3](=O)[O][CX3](=O)",
        "a_requires": "[CX3](=O)[OH]", "x_requires": "[CX3](=O)[OH]",
    },
    "acetal": {
        "smarts": "[CH1:1]=[O:2].[OH:3][C:4].[OH:5][C:6]>>[CH1:1]([O:3][C:4])[O:5][C:6]",
        "water_loss": 36.04, "product_check": "[CX4]([O])([O])",
        "a_requires": "[CH1]=[O]", "x_requires": "[OH]",
    },
    "mercaptal": {
        "smarts": "[CH1:1]=[O:2].[SH:3][C:4].[SH:5][C:6]>>[CH1:1]([S:3][C:4])[S:5][C:6]",
        "water_loss": 36.04, "product_check": "[CX4]([S])([S])",
        "a_requires": "[CH1]=[O]", "x_requires": "[SH]",
    },
    "ether": {
        "smarts": "[OH:1][C:2].[OH:3][C:4]>>[C:2][O:1][C:4]",
        "water_loss": 18.02, "product_check": "[OD2]([C])[C]",
        "a_requires": "[OH]", "x_requires": "[OH]",
    },
    "imide": {
        "smarts": "[C:1](=[O:2])[OH:3].[NH2:4]>>[C:1](=[O:2])[NH:4]",
        "water_loss": 18.02, "product_check": "[CX3](=O)[N]",
        "a_requires": "[CX3](=O)[OH]", "x_requires": "[NH2]",
    },
}

COMPILED_REACTIONS = {}
for ptype, info in REACTION_TEMPLATES.items():
    try:
        rxn = AllChem.ReactionFromSmarts(info['smarts'])
        COMPILED_REACTIONS[ptype] = rxn
        print(f"  ✓ {ptype}")
    except Exception as e:
        print(f"  ✗ {ptype}: {e}")
print(f"{len(COMPILED_REACTIONS)} templates compiled.")

  ✓ ester
  ✓ oxime_ester
  ✓ amide
  ✓ anhydride
  ✓ acetal
  ✓ mercaptal
  ✓ ether
  ✓ imide
8 templates compiled.


## 8. Guard Stack

In [24]:
def guard_inputs(smi_a, smi_x, product_type):
    issues = []
    mol_a = Chem.MolFromSmiles(smi_a) if smi_a else None
    mol_x = Chem.MolFromSmiles(smi_x) if smi_x else None
    if mol_a is None:
        return False, ["Component A: invalid or missing SMILES"], None, None
    if mol_x is None:
        return False, ["Component X: invalid or missing SMILES"], mol_a, None

    template = REACTION_TEMPLATES.get(product_type)
    if template:
        a_pat = Chem.MolFromSmarts(template['a_requires'])
        if a_pat and not mol_a.HasSubstructMatch(a_pat):
            issues.append(f"A lacks: {template['a_requires']}")
        x_pat = Chem.MolFromSmarts(template['x_requires'])
        if x_pat and not mol_x.HasSubstructMatch(x_pat):
            # Check for oxime reroute
            if product_type == 'ester':
                oxime_pat = Chem.MolFromSmarts('[OX2H1][NX2]')
                if oxime_pat and mol_x.HasSubstructMatch(oxime_pat):
                    return True, ['REROUTE_OXIME_ESTER'], mol_a, mol_x
            issues.append(f"X lacks: {template['x_requires']}")

    if issues:
        return False, issues, mol_a, mol_x
    return True, [], mol_a, mol_x


def guard_reaction(rxn, mol_a, mol_x, product_type):
    products = None
    try:
        products = rxn.RunReactants((mol_a, mol_x))
    except:
        pass
    if not products:
        try:
            products = rxn.RunReactants((mol_x, mol_a))
        except:
            pass
    if not products:
        return None, "No products generated"

    valid = []
    for ps in products:
        for p in ps:
            try:
                Chem.SanitizeMol(p)
                s = Chem.MolToSmiles(p)
                if '.' not in s:
                    valid.append(s)
            except:
                pass
    if not valid:
        return None, "All products invalid or disconnected"
    unique = list(set(valid))
    if len(unique) == 1:
        return unique[0], "OK"
    best = max(unique, key=lambda s: Chem.MolFromSmiles(s).GetNumHeavyAtoms())
    return best, f"WARNING: {len(unique)} products, picked largest"


def guard_output(smi_product, smi_a, smi_x, product_type):
    issues = []
    mol = Chem.MolFromSmiles(smi_product)
    mol_a = Chem.MolFromSmiles(smi_a)
    mol_x = Chem.MolFromSmiles(smi_x)
    if mol is None:
        return "FAIL", ["Product SMILES invalid"]

    mw_prod = Descriptors.MolWt(mol)
    mw_a = Descriptors.MolWt(mol_a)
    mw_x = Descriptors.MolWt(mol_x)
    template = REACTION_TEMPLATES.get(product_type, {})
    expected_loss = template.get('water_loss', 18.02)
    expected_mw = mw_a + mw_x - expected_loss
    if abs(mw_prod - expected_mw) > 3.0:
        issues.append(f"MW: expected {expected_mw:.1f}, got {mw_prod:.1f}")

    if 'product_check' in template:
        pat = Chem.MolFromSmarts(template['product_check'])
        if pat and not mol.HasSubstructMatch(pat):
            issues.append(f"Product lacks {product_type} group")

    if not issues:
        return "PASS", []
    return "FAIL" if any(not i.startswith("WARNING") for i in issues) else "WARNING", issues


def guard_simple_name(smi, name):
    """FIX #6: includes sulfonamide check."""
    issues = []
    if smi is None:
        return "FAILED", ["No SMILES"]
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return "FAIL", ["Invalid SMILES"]
    if '.' in smi:
        issues.append(f"MIXTURE: {len(smi.split('.'))} disconnected fragments")
    ha = mol.GetNumHeavyAtoms()
    if ha < 3 and len(name.split()) > 2:
        issues.append(f"SUSPICIOUSLY SMALL: {ha} heavy atoms")

    low = name.lower()
    if 'ester' in low:
        if not mol.HasSubstructMatch(Chem.MolFromSmarts('[CX3](=O)[OX2][C]')):
            issues.append("Name says 'ester' but no ester group")
    if 'amide' in low or 'anilide' in low:
        # FIX #6: Check both carboxamide AND sulfonamide
        has_carboxamide = mol.HasSubstructMatch(Chem.MolFromSmarts('[CX3](=O)[NX3]'))
        has_sulfonamide = mol.HasSubstructMatch(Chem.MolFromSmarts('[SX4](=O)(=O)[NX3]'))
        if not has_carboxamide and not has_sulfonamide:
            issues.append("Name says 'amide' but no amide/sulfonamide group")
    if 'ether' in low:
        if not mol.HasSubstructMatch(Chem.MolFromSmarts('[OD2]([C])[C]')):
            issues.append("Name says 'ether' but no ether group")

    if not issues:
        return "PASS", []
    has_err = any('MIXTURE' in i or 'SMALL' in i or 'no ester' in i or 'no amide' in i or 'no ether' in i for i in issues)
    return "FAIL" if has_err else "WARNING", issues


print("Guard stack v4 ready.")

Guard stack v4 ready.


## 9. Sub-Derivatization + Master Resolver

In [25]:
def resolve_modifier_x(modifier_name, sub_deriv=None, base_compound=None):
    if sub_deriv is None or base_compound is None:
        smi, source = resolve_name_to_smiles(modifier_name)
        return smi, source, None
    base_smi, base_source = resolve_name_to_smiles(base_compound)
    if base_smi is None:
        smi, source = resolve_name_to_smiles(modifier_name)
        return smi, source, f"base '{base_compound}' failed"
    sub_info = SUB_DERIVATIZATIONS.get(sub_deriv)
    if sub_info is None:
        return base_smi, base_source, f"unknown sub-deriv: {sub_deriv}"
    try:
        rxn = AllChem.ReactionFromSmarts(sub_info['template'])
        base_mol = Chem.MolFromSmiles(base_smi)
        if base_mol is None:
            return None, 'failed', 'base SMILES invalid'
        products = rxn.RunReactants((base_mol,))
        if products:
            prod = products[0][0]
            Chem.SanitizeMol(prod)
            return Chem.MolToSmiles(prod), f"{base_source}+{sub_deriv}", f"applied {sub_deriv}"
        return base_smi, base_source, f"{sub_deriv} template didn't fire"
    except Exception as e:
        return base_smi, base_source, f"{sub_deriv} error: {e}"


def resolve_compound(name, route, route_details):
    result = {
        'smiles': None, 'route': route, 'source': None,
        'status': None, 'guard_status': None, 'guard_notes': '',
        'parsed_A': None, 'parsed_X': None, 'parsed_Y': None,
        'smiles_A': None, 'smiles_X': None,
    }

    if route == 'SKIP':
        result['status'] = 'IRREGULAR_SKIPPED'
        result['guard_notes'] = route_details.get('reason', 'irregular')
        return result

    # Apply typo fix to name
    name_fixed = fix_typos(name)

    if route == 'SIMPLE':
        smi, source = resolve_name_to_smiles(name_fixed)
        result['smiles'] = smi
        result['source'] = source
        if smi is None:
            result['status'] = 'FAILED_SIMPLE'
            result['guard_status'] = 'NO_SMILES'
        else:
            gs, gi = guard_simple_name(smi, name_fixed)
            result['guard_status'] = gs
            result['guard_notes'] = '; '.join(gi)
            if gs == 'PASS':
                result['status'] = 'RESOLVED_SIMPLE'
            elif gs == 'WARNING':
                result['status'] = 'RESOLVED_SIMPLE_WARNING'
            else:
                if 'MIXTURE' in result['guard_notes']:
                    retry = _resolve_reaction(name_fixed, route_details)
                    if retry['smiles']:
                        retry['guard_notes'] = f"REROUTED (mixture); {retry['guard_notes']}"
                        return retry
                result['status'] = 'RESOLVED_SIMPLE_FLAGGED'
        return result

    if route == 'REACTION':
        return _resolve_reaction(name_fixed, route_details)

    result['status'] = 'UNKNOWN_ROUTE'
    return result


def _resolve_reaction(name, route_details):
    result = {
        'smiles': None, 'route': 'REACTION', 'source': None,
        'status': None, 'guard_status': None, 'guard_notes': '',
        'parsed_A': None, 'parsed_X': None, 'parsed_Y': None,
        'smiles_A': None, 'smiles_X': None,
    }

    parsed = parse_AXY(name)
    
    # ✅ Add this block right here
    if parsed['status'] == 'NO_MATCH':
        result['status'] = 'FAILED_PARSE'
        result['guard_status'] = 'FAILED_PARSE'
        result['guard_notes'] = f"Could not parse A–X–Y structure for: {name}"
        return result
    
    result['parsed_A'] = parsed['parent_A']
    result['parsed_X'] = parsed['modifier_X']
    result['parsed_Y'] = parsed['product_type_Y']
    product_type = parsed['product_type_Y']

    # Try direct first
    full_smi, full_src = resolve_name_to_smiles(name)
    if full_smi:
        gs, gi = guard_simple_name(full_smi, name)
        if gs == 'PASS':
            result['smiles'] = full_smi
            result['source'] = f"{full_src}_direct"
            result['status'] = 'RESOLVED_DIRECT'
            result['guard_status'] = 'PASS'
            return result

    # Trade name
    if parsed['parenthetical']:
        t_smi, t_src = resolve_name_to_smiles(parsed['parenthetical'])
        if t_smi:
            gs, gi = guard_simple_name(t_smi, name)
            if gs == 'PASS':
                result['smiles'] = t_smi
                result['source'] = f"{t_src}_tradename"
                result['status'] = 'RESOLVED_TRADENAME'
                result['guard_status'] = 'PASS'
                return result

    # Resolve components
    if parsed['parent_A'] is None:
        result['status'] = 'FAILED_PARSE'
        result['guard_notes'] = 'Could not parse parent A'
        return result

    smi_a, src_a = resolve_name_to_smiles(parsed['parent_A'])
    result['smiles_A'] = smi_a

    if parsed['modifier_X'] is None:
        result['status'] = 'FAILED_PARSE'
        result['guard_notes'] = 'Could not parse modifier X'
        return result

    smi_x, src_x, x_note = resolve_modifier_x(
        parsed['modifier_X'],
        sub_deriv=parsed['x_sub_derivatization'],
        base_compound=parsed['x_base_compound']
    )
    result['smiles_X'] = smi_x

    # Salt
    if product_type == 'salt':
        if smi_a:
            deprot = smi_a.replace('C(=O)O', 'C(=O)[O-]', 1).replace('C(O)=O', 'C([O-])=O', 1)
            result['smiles'] = deprot
            result['source'] = f"{src_a}_salt_ionic"
            result['status'] = 'RESOLVED_SALT'
            result['guard_status'] = 'WARNING'
            result['guard_notes'] = f"Salt; cation={parsed['modifier_X']}"
            return result
        result['status'] = 'FAILED_SALT'
        return result

    # Unknown product types
    if product_type in ('unknown', 'unknown_comma_acid', 'entity_scan_detected', 'suffix_scan_detected'):
        # Try direct IUPAC-style lookup of the reconstructed name
        if smi_a and parsed['modifier_X']:
            # Try "modifier parent" format (e.g., "ethyl benzoate")
            acid_key = parsed['parent_A'].lower().replace(' acid', '').strip()
            ester_suffix = ACID_TO_ESTER.get(acid_key)
            if ester_suffix and parsed['modifier_X']:
                reconstructed = f"{parsed['modifier_X']} {ester_suffix}"
                r_smi, r_src = resolve_name_to_smiles(reconstructed)
                if r_smi:
                    gs, gi = guard_simple_name(r_smi, name)
                    if gs == 'PASS':
                        result['smiles'] = r_smi
                        result['source'] = f"{r_src}_reconstructed"
                        result['status'] = 'RESOLVED_RECONSTRUCTED'
                        result['guard_status'] = 'PASS'
                        return result
        result['status'] = 'FAILED_NO_TEMPLATE'
        result['guard_notes'] = f'No template for: {product_type}'
        return result

    if product_type not in COMPILED_REACTIONS:
        result['status'] = 'FAILED_NO_TEMPLATE'
        result['guard_notes'] = f'No compiled template: {product_type}'
        return result

    # Input guards
    input_ok, input_issues, mol_a, mol_x = guard_inputs(smi_a, smi_x, product_type)
    if input_issues and 'REROUTE_OXIME_ESTER' in input_issues:
        product_type = 'oxime_ester'
        input_ok = True
        input_issues = []
    if not input_ok:
        result['status'] = 'FAILED_INPUT_GUARD'
        result['guard_status'] = 'FAIL'
        result['guard_notes'] = '; '.join(input_issues)
        return result

    # Run reaction
    rxn = COMPILED_REACTIONS[product_type]
    if parsed['multiplicity'] == 1:
        prod_smi, rxn_msg = guard_reaction(rxn, mol_a, mol_x, product_type)
    elif parsed['multiplicity'] >= 2:
        prod_smi, rxn_msg = guard_reaction(rxn, mol_a, mol_x, product_type)
        if prod_smi:
            for _ in range(parsed['multiplicity'] - 1):
                mol_prod = Chem.MolFromSmiles(prod_smi)
                if mol_prod:
                    p2, m2 = guard_reaction(rxn, mol_a, mol_prod, product_type)
                    if p2:
                        prod_smi = p2
                        rxn_msg += f"; +1: {m2}"
    else:
        prod_smi, rxn_msg = guard_reaction(rxn, mol_a, mol_x, product_type)
        rxn_msg += " (polymer unit)"

    if prod_smi is None:
        result['status'] = 'FAILED_REACTION'
        result['guard_status'] = 'FAIL'
        result['guard_notes'] = rxn_msg
        return result

    out_status, out_issues = guard_output(prod_smi, smi_a, smi_x, product_type)
    result['smiles'] = prod_smi
    result['source'] = f"reaction_{product_type}"
    result['guard_status'] = out_status
    result['guard_notes'] = '; '.join(out_issues) if out_issues else rxn_msg
    if out_status == 'PASS':
        result['status'] = 'RESOLVED_REACTION'
    elif out_status == 'WARNING':
        result['status'] = 'RESOLVED_REACTION_WARNING'
    else:
        result['status'] = 'RESOLVED_REACTION_FLAGGED'
    return result


print("Master resolver v4 ready.")

Master resolver v4 ready.


## 11. Results Summary

In [ ]:
results_df = pd.DataFrame(results)
for col in results_df.columns:
    df[col] = results_df[col].values

n_resolved = df['smiles'].notna().sum()
print(f"\nOVERALL: {n_resolved}/{len(df)} resolved ({n_resolved/len(df)*100:.1f}%)")

print(f"\nBy route:")
for route in ['SIMPLE', 'REACTION', 'SKIP']:
    subset = df[df['_route'] == route]
    if len(subset) > 0:
        resolved = subset['smiles'].notna().sum()
        print(f"  {route:10s}: {resolved}/{len(subset)} ({resolved/len(subset)*100:.1f}%)")

print(f"\nBy source:")
for src, count in df['source'].value_counts(dropna=False).items():
    print(f"  {str(src):35s} {count:5d}")

print(f"\nGuard status:")
for gs, count in df['guard_status'].value_counts(dropna=False).items():
    print(f"  {str(gs):20s} {count:5d}")

## 12. Flagged Compounds

In [ ]:
flagged = df[df['guard_status'].isin(['FAIL', 'WARNING'])]
print(f"Flagged compounds: {len(flagged)}")
print(f"\nSample flagged (first 20):")
print("=" * 80)
for _, row in flagged.head(20).iterrows():
    print(f"  {row['_name'][:65]}")
    print(f"    Route: {row['_route']}  Status: {row['status']}")
    print(f"    Guard: {row['guard_status']}  Notes: {str(row['guard_notes'])[:80]}")
    if row.get('parsed_A'):
        print(f"    Parsed: A={row['parsed_A']}  X={row['parsed_X']}  Y={row['parsed_Y']}")
    if row['smiles']:
        print(f"    SMILES: {str(row['smiles'])[:60]}")
    print()

## 13. Export

In [ ]:
OUTPUT_FILE = "pass1_v4_resolved.xlsx"

export_cols = [NAME_COL] + [c for c in df.columns if not c.startswith('_') and c != NAME_COL]
seen = set()
clean_cols = [c for c in export_cols if c in df.columns and not (c in seen or seen.add(c))]
df_export = df[clean_cols].copy()

with pd.ExcelWriter(OUTPUT_FILE) as writer:
    df_export.to_excel(writer, sheet_name='All_Data', index=False)
    df_export[df_export['smiles'].notna()].to_excel(writer, sheet_name='Resolved', index=False)
    df_export[df_export['status'].astype(str).str.contains('FAILED', na=False)].to_excel(writer, sheet_name='Failed', index=False)
    df_export[df_export['guard_status'].isin(['FAIL', 'WARNING'])].to_excel(writer, sheet_name='Flagged', index=False)
    df_export[df_export['source'].astype(str).str.contains('reaction_', na=False)].to_excel(writer, sheet_name='Reaction_Built', index=False)
    df_export[df_export['status'] == 'IRREGULAR_SKIPPED'].to_excel(writer, sheet_name='Irregular', index=False)

print(f"Exported: {OUTPUT_FILE}")
for sheet in ['All_Data', 'Resolved', 'Failed', 'Flagged', 'Reaction_Built', 'Irregular']:
    try:
        n = len(pd.read_excel(OUTPUT_FILE, sheet_name=sheet))
        print(f"  {sheet}: {n}")
    except:
        pass

colab_files.download(OUTPUT_FILE)
print(f"\n>>> Downloaded: {OUTPUT_FILE}")